In [ ]:
# Download Kaggle Dataset

In [30]:
import pandas as pd
from bs4 import BeautifulSoup

ans_df = pd.read_csv("archive/Answers.csv", encoding="latin1")
ques_df = pd.read_csv("archive/Questions.csv", encoding="latin1")
tags_df = pd.read_csv("archive/Tags.csv", encoding="latin1")

In [31]:
print(f"Questions columns: {ques_df.columns}")
print(f"Answer columns: {ans_df.columns}")
print(f"Tags columns: {tags_df.columns}")

Questions columns: Index(['Id', 'OwnerUserId', 'CreationDate', 'Score', 'Title', 'Body'], dtype='object')
Answer columns: Index(['Id', 'OwnerUserId', 'CreationDate', 'ParentId', 'Score', 'Body'], dtype='object')
Tags columns: Index(['Id', 'Tag'], dtype='object')


In [32]:
def clean_html(text):
    if not isinstance(text, str):
        return ""

    return BeautifulSoup(
        text,
        "html.parser"
    ).get_text(
        separator=" ",
        strip=True
    )

In [33]:
tags_agg = (
    tags_df
    .groupby("Id")["Tag"]
    .apply(
        lambda x: ", ".join(
            str(tag)
            for tag in x
            if pd.notna(tag)
        )
    )
    .reset_index(name="Tags")
)

In [34]:
answers_agg = (
    ans_df
    .groupby("ParentId")
    .agg({
        "Body": list,
        "Score": list
    })
    .reset_index()
)

In [35]:
final_df = (
    ques_df
    .merge(tags_agg, on="Id", how="left")
    .merge(
        answers_agg,
        left_on="Id",
        right_on="ParentId",
        how="left"
    )
)

In [36]:
final_df = final_df.rename(
    columns={
        "Score_x": "question_score",
        "Score_y": "answer_scores",
        "Body_x": "question_body",
        "Body_y": "answer_bodies"
    }
)

In [37]:
def get_top_3_answers(row):

    scores = row["answer_scores"]
    answers = row["answer_bodies"]

    if not isinstance(scores, list):
        return ""

    if not isinstance(answers, list):
        return ""

    pairs = sorted(
        zip(scores, answers),
        key=lambda x: x[0],
        reverse=True
    )[:3]

    formatted = []

    for rank, (score, answer) in enumerate(pairs, start=1):

        formatted.append(
            f"""
Answer {rank}
Score: {score}

{clean_html(answer)}
"""
        )

    return "\n".join(formatted)

In [38]:
final_df["top_3_answers"] = (
    final_df.apply(
        get_top_3_answers,
        axis=1
    )
)

In [ ]:
final_df["question_text"] = (
    final_df["question_body"]
    .apply(clean_html)
)

In [ ]:
final_df["document"] = (
    "Title: "
    + final_df["Title"].fillna("")
    + "\n\nTags: "
    + final_df["Tags"].fillna("")
    + "\n\nQuestion:\n"
    + final_df["question_text"].fillna("")
    + "\n\nTop Answers:\n"
    + final_df["top_3_answers"].fillna("")
)

In [ ]:
final_df = final_df[
    final_df["top_3_answers"].str.len() > 0
]

final_df = final_df[
    final_df["question_score"] >= 2
]

In [ ]:
rag_df = final_df[
    [
        "Id",
        "Title",
        "Tags",
        "question_score",
        "document"
    ]
]

In [ ]:
rag_df.to_parquet(
    "stack_overflow_rag.parquet",
    index=False
)

In [ ]:
len(rag_df)